In [1]:
import pandas as pd
import glob


In [2]:
import glob
import pandas as pd

PATH = r"C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw/"

files = glob.glob(PATH + "CRMLSSold*")

print(f"Number of monthly SOLD files found: {len(files)}")

row_counts = []
for f in files:
    temp = pd.read_csv(f, encoding='latin1')
    row_counts.append((f, len(temp)))

print("\nRow counts BEFORE concatenation:")
for fname, count in row_counts:
    print(f"{fname}: {count:,} rows")

df_sold = pd.concat(
    [pd.read_csv(f, encoding='latin1') for f in files],
    ignore_index=True
)

print(f"\nTotal rows AFTER concatenation: {len(df_sold):,}")

df_res_sold = df_sold[df_sold["PropertyType"] == "Residential"]

print(f"Total rows AFTER Residential filter: {len(df_res_sold):,}")

df_res_sold.to_csv("combined_sold.csv", index=False)
print("\nSaved filtered SOLD dataset to combined_sold.csv")



Number of monthly SOLD files found: 27


C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:12: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')
C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')
C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:12: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(f, encoding='latin1')



Row counts BEFORE concatenation:
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202401.csv: 17,976 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202402.csv: 19,925 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202403.csv: 23,276 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202404.csv: 24,640 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202405.csv: 26,487 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202406.csv: 24,328 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202407.csv: 26,240 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202408.csv: 24,558 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202409.csv: 21,267 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202410.csv: 23,274 rows
C:/Users/alexa/OneDrive/Desktop/IDX Internship/raw\CRMLSSold202411.csv: 20,279 rows
C:/Users/alexa/OneDrive/Desktop/IDX Intern

C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:20: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],
C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:20: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],
C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\314879099.py:20: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f, encoding='latin1') for f in files],



Total rows AFTER concatenation: 587,279
Total rows AFTER Residential filter: 394,150

Saved filtered SOLD dataset to combined_sold.csv


In [3]:
df_res_sold.shape
df_res_sold.info()


<class 'pandas.core.frame.DataFrame'>
Index: 394150 entries, 0 to 587269
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 345454 non-null  object 
 1   ListAgentAOR                  347965 non-null  object 
 2   Flooring                      252658 non-null  object 
 3   ViewYN                        360505 non-null  object 
 4   WaterfrontYN                  255 non-null     object 
 5   BasementYN                    7723 non-null    object 
 6   PoolPrivateYN                 359983 non-null  object 
 7   OriginalListPrice             393435 non-null  float64
 8   ListingKey                    394150 non-null  int64  
 9   ListAgentEmail                368591 non-null  object 
 10  CloseDate                     394150 non-null  object 
 11  ClosePrice                    394148 non-null  float64
 12  ListAgentFirstName            391191 non-null  ob

In [4]:
print("Initial row count:", len(df_res_sold))
print("Initial column count:", df_res_sold.shape[1])

# 1. DOCUMENT UNIQUE PROPERTY TYPES
print("\nUnique Property Types:")
print(df_sold["PropertyType"].unique())

# 2. NULL COUNT SUMMARY TABLE
null_summary = df_res_sold.isnull().sum().to_frame(name="NullCount")
null_summary["NullPercent"] = (null_summary["NullCount"] / len(df_res_sold)) * 100

print("\nNull Count Summary Table:")
print(null_summary)

# 3. FLAG COLUMNS ABOVE 90% NULL
high_null_cols = null_summary[null_summary["NullPercent"] > 90].index.tolist()

print("\nColumns ABOVE 90% null:")
for col in high_null_cols:
    print(f"- {col}")

# 4. REMOVE COLUMNS ABOVE 90% NULL
df_filtered_sold = df_res_sold.drop(columns=high_null_cols)
print(f"\nColumn count AFTER removing >90% null columns: {df_filtered_sold.shape[1]}")

# 5. NUMERIC DISTRIBUTION SUMMARY
#    For ClosePrice, LivingArea, DaysOnMarket
numeric_cols = ["ClosePrice", "LivingArea", "DaysOnMarket"]

print("\nNumeric Distribution Summary:")
for col in numeric_cols:
    if col in df_filtered_sold.columns:
        print(f"\n--- {col} ---")
        print(df_filtered_sold[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    else:
        print(f"\n--- {col} NOT FOUND IN DATASET ---")

# 6. SAVE FILTERED DATASET
df_filtered_sold.to_csv("filtered_week2_3_sold_output.csv", index=False)
print("\nSaved cleaned dataset to filtered_week2_3_sold_output.csv")

Initial row count: 394150
Initial column count: 84

Unique Property Types:
['Residential' 'CommercialLease' 'Land' 'ResidentialLease'
 'ManufacturedInPark' 'ResidentialIncome' 'CommercialSale'
 'BusinessOpportunity']

Null Count Summary Table:
                             NullCount  NullPercent
BuyerAgentAOR                    48696    12.354687
ListAgentAOR                     46185    11.717620
Flooring                        141492    35.898008
ViewYN                           33645     8.536090
WaterfrontYN                    393895    99.935304
...                                ...          ...
OriginatingSystemSubName        358351    90.917417
BuyerAgencyCompensationType     348014    88.294812
BuyerAgencyCompensation         348025    88.297602
latfilled                       330266    83.791957
lonfilled                       330266    83.791957

[84 rows x 2 columns]

Columns ABOVE 90% null:
- WaterfrontYN
- BasementYN
- FireplacesTotal
- AboveGradeFinishedArea
- TaxAnnualAm

In [5]:

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

df_filtered_sold['year_month'] = pd.to_datetime(
    df_filtered_sold['ListingContractDate']
).dt.to_period('M')

sold_with_rates = df_filtered_sold.merge(
    mortgage_monthly, on='year_month', how='left'
)

null_rates = sold_with_rates['rate_30yr_fixed'].isnull().sum()
print("Sold rows with NULL mortgage rate:", null_rates)

sold_with_rates.to_csv("sold_with_mortgage_rates.csv", index=False)
print("Saved sold_with_mortgage_rates.csv")


Sold rows with NULL mortgage rate: 1
Saved sold_with_mortgage_rates.csv


In [6]:

# 1. LOAD EXISTING LISTINGS CSV (EDIT IN PLACE)

print("Initial row count:", len(sold_with_rates))
print("Initial column count:", sold_with_rates.shape[1])

# 2. DATA TYPE CONFIRMATION

print("\nData types BEFORE cleaning:")
print(sold_with_rates.dtypes)

# 3. DATE CONSISTENCY CHECKS
date_cols = ["ListingContractDate", "CloseDate"]

for col in date_cols:
    if col in sold_with_rates.columns:
        sold_with_rates[col] = pd.to_datetime(sold_with_rates[col], errors="coerce")

print("\nInvalid date counts:")
for col in date_cols:
    if col in sold_with_rates.columns:
        invalid = sold_with_rates[col].isnull().sum()
        print(f"{col}: {invalid} invalid entries")


# 4. GEOGRAPHIC DATA QUALITY CHECK

lat_col = "Latitude"
lon_col = "Longitude"

invalid_lat = sold_with_rates[(sold_with_rates[lat_col] < -90) | (sold_with_rates[lat_col] > 90)].shape[0] if lat_col in sold_with_rates else 0
invalid_lon = sold_with_rates[(sold_with_rates[lon_col] < -180) | (sold_with_rates[lon_col] > 180)].shape[0] if lon_col in sold_with_rates else 0

print("\nGeographic Data Quality Summary:")
print(f"Invalid Latitude values: {invalid_lat}")
print(f"Invalid Longitude values: {invalid_lon}")

if lat_col in sold_with_rates and lon_col in sold_with_rates:
    df = sold_with_rates[
        (sold_with_rates[lat_col].between(-90, 90)) &
        (sold_with_rates[lon_col].between(-180, 180))
    ]

print("Rows AFTER removing invalid coordinates:", len(df))

# 5. FINAL DATA TYPE CHECK

print("\nData types AFTER cleaning:")
print(sold_with_rates.dtypes)

# 6. SAVE CLEANED, ANALYSIS-READY DATASET

sold_with_rates.to_csv("sold_cleaned_final.csv", index=False)

print("\nSaved sold_cleaned_final.csv")

Initial row count: 394150
Initial column count: 69

Data types BEFORE cleaning:
BuyerAgentAOR                 object
ListAgentAOR                  object
Flooring                      object
ViewYN                        object
PoolPrivateYN                 object
                             ...    
BuyerAgencyCompensation      float64
latfilled                     object
lonfilled                     object
year_month                 period[M]
rate_30yr_fixed              float64
Length: 69, dtype: object

Invalid date counts:
ListingContractDate: 1 invalid entries
CloseDate: 0 invalid entries

Geographic Data Quality Summary:
Invalid Latitude values: 1
Invalid Longitude values: 1
Rows AFTER removing invalid coordinates: 378350

Data types AFTER cleaning:
BuyerAgentAOR                 object
ListAgentAOR                  object
Flooring                      object
ViewYN                        object
PoolPrivateYN                 object
                             ...    
BuyerAgenc

In [7]:
import pandas as pd

df = pd.read_csv("../raw/sold_cleaned_final.csv")

df["PriceRatio"] = df["ClosePrice"] / df["OriginalListPrice"]

df["CloseToOriginalListRatio"] = df["ClosePrice"] / df["OriginalListPrice"]

df["PPSF"] = df["ClosePrice"] / df["LivingArea"]

df["DaysOnMarket"] = df["DaysOnMarket"]

df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["CloseYear"] = df["CloseDate"].dt.year
df["CloseMonth"] = df["CloseDate"].dt.month
df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

df["ListingToContractDays"] = (
    pd.to_datetime(df["PurchaseContractDate"]) -
    pd.to_datetime(df["ListingContractDate"])
).dt.days

df["ContractToCloseDays"] = (
    pd.to_datetime(df["CloseDate"]) -
    pd.to_datetime(df["PurchaseContractDate"])
).dt.days

sample_output = df[[
    "ClosePrice", "OriginalListPrice", "LivingArea",
    "PriceRatio", "CloseToOriginalListRatio", "PPSF",
    "DaysOnMarket", "YrMo",
    "ListingToContractDays", "ContractToCloseDays",
    "PropertyType", "CountyOrParish"
]].head(10)

print("\n=== SAMPLE OUTPUT TABLE ===")
print(sample_output)

segment_summary = df.groupby("PropertyType").agg({
    "ClosePrice": "median",
    "PPSF": "median",
    "DaysOnMarket": "median",
    "PriceRatio": "median",
    "ListingToContractDays": "median",
    "ContractToCloseDays": "median"
}).reset_index()

print("\n=== SEGMENT SUMMARY BY PROPERTY TYPE ===")
print(segment_summary)

sample_output.to_csv("week6_sample_output_sold.csv", index=False)
segment_summary.to_csv("week6_segment_summary_sold.csv", index=False)

print("\nWeek 6 feature engineering complete.")

C:\Users\alexa\AppData\Local\Temp\ipykernel_10728\4067403576.py:3: DtypeWarning: Columns (0,1,7,51,63,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../raw/sold_cleaned_final.csv")



=== SAMPLE OUTPUT TABLE ===
   ClosePrice  OriginalListPrice  LivingArea  PriceRatio  \
0    240000.0           499000.0      1140.0    0.480962   
1    815000.0           759900.0      1974.0    1.072510   
2    810000.0           739900.0      1974.0    1.094743   
3    858000.0                NaN      1995.0         NaN   
4   1890500.0          1890500.0      3194.0    1.000000   
5   2100000.0          2100000.0      3736.0    1.000000   
6   1950000.0          1950000.0      2100.0    1.000000   
7   2340000.0                NaN      2442.0         NaN   
8   1485000.0          1550000.0      1601.0    0.958065   
9   1130000.0           999000.0      2136.0    1.131131   

   CloseToOriginalListRatio        PPSF  DaysOnMarket     YrMo  \
0                  0.480962  210.526316           777  2024-01   
1                  1.072510  412.867275            33  2024-01   
2                  1.094743  410.334347           228  2024-01   
3                       NaN  430.075188       